In [1]:
from sedona.spark import SedonaContext
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/18 20:37:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/18 20:37:23 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/18 20:37:23 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/18 20:37:23 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/18 20:37:23 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.geometryObjects.Geography, which is already registered.
25/08/18 20:37:23 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/18 20:37:23 WARN SimpleFunctionRegistry: The function st_env

# nested loop join

In [3]:
places = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/places")

In [4]:
places.count()

4694140

In [16]:
sample = places.select("id", "geometry").sample(0.0003)
left = sample.collect()
right = left

In [17]:
def match(left, right):
    return l.geometry.buffer(0.001).intersects(right.geometry) and left.id != right.id

In [18]:
result = []
for l in left:
    for r in right:
        if not match(l, r):
            continue
 
        result.append((l, r))

In [19]:
len(result)

6

# Cartesian Product Join

In [20]:
from shapely.wkt import loads
import pyspark.sql.types as t
import pyspark.sql.functions as f

In [21]:
line = loads("LINESTRING (-46.988525 -23.344778, -46.771545 -23.569022, -46.149445 -23.732555)")

point = loads("POINT(-46.988525 -23.344778)")

points = sedona.createDataFrame([[1, point]])\
    .selectExpr("_1 AS id", "_2 as geom")

lines = sedona.createDataFrame([[1, line]])\
    .selectExpr("_1 AS id", "_2 as geom")

In [22]:
intersects = f.udf(
    lambda left, right: left.intersects(right),
    t.BooleanType()
)
 
result = points\
    .alias("p")\
    .join(
        lines.alias("l"),
        intersects(f.col("p.geom"),
        f.col("l.geom"))
    )

# Spatial Join

In [23]:
# optimized join snippet, using r index

In [24]:
import rtree

In [25]:
r_index = rtree.Rtree()
index = 0
left_mapping = {}

for l in left:
    left_mapping[index] = l
    minx, miny, maxx, maxy = l.geometry.buffer(0.001).bounds

    r_index.add(index, [minx, miny, maxx, maxy])

    index += 1


def get_get_bounds(geom):
    return geom.geometry.bounds

def predicate(l, r):
    return left_mapping[l].geometry.buffer(0.001).intersects(r.geometry) and left_mapping[l].id != r.id

In [26]:
result_index = []
 
for r in right:
    candidates = r_index.intersection(get_get_bounds(r))
    candidates_filtered = [[c, r] for c in candidates if predicate(c, r)]
    #  if predicate(c, r)
    if len(candidates_filtered) > 1:
        print(len(candidates_filtered))
    result_index.extend(candidates_filtered)

In [27]:
len(result_index)

6